## Gold Layer

In [0]:
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta import *

In [0]:
## first time full load to be done in SAles Fact Table from SILVER------------>>GOLD

sales_df=spark.sql("select * from retailfashiondata.silvertransformed.sales_fact")
display(sales_df)

## intial load done using the full load.

# (sales_df.write.format("delta")
#               .mode("overwrite")
#               .saveAsTable("retailfashiondata.gold.sales_fact"))

In [0]:
# Get the max ingestion_time from the target (gold) table
last_modified_date = spark.sql("SELECT max(ingestion_time) FROM retailfashiondata.gold.sales_fact").collect()[0][0]


sales_fact= spark.sql("select * from retailfashiondata.silvertransformed.sales_fact")


sales_fact= sales_fact.filter(col("ingestion_time")>last_modified_date)

display(sales_fact)

(sales_fact.write.format("delta").mode("append").saveAsTable("retailfashiondata.gold.sales_fact"))




In [0]:
%sql
MERGE INTO retailfashiondata.gold.sales_fact AS t
USING sales_fact s
ON t.transaction_id = s.transaction_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

In [0]:
%sql
Select * from  retailfashiondata.gold.sales_fact;

In [0]:
#create table cust_dim

spark.sql("""Create Table if not exists retailfashiondata.gold.cust_dim(
customer_id string,
age integer,
gender string,
city string,
email string,
ingestion_time timestamp,
start_date date,
end_date date,
is_current string)
          Using DELTA""")


# # create table prod_dim

# spark.sql("""Create Table if not exists retailfashiondata.gold.prod_dim(
# product_id string,
# category string,
# color string,
# size string,
# season string,
# supplier string,
# cost_price double,
# list_price double,
# ingestion_time timestamp,
# profit_margin double,
# start_date date,
# end_date date,
# is_current string)
#           Using DELTA""")

# # # create store dim

# spark.sql("""Create Table if not exists retailfashiondata.gold.store_dim(
# store_id string,
# store_name string,
# region string,
# store_size_m2 integer,
# ingestion_time timestamp,
# start_date date,
# end_date date,
# is_current string)
#           Using DELTA""")


# # create sales fact

# spark.sql("""Create Table if not exists retailfashiondata.gold.sales_fact(
# transaction_id string,
# date date,
# product_id string,
# store_id string,
# customer_id string,
# quantity integer,
# discount double,
# returned boolean,
# ingestion_time timestamp
# )USING DELTA""")



In [0]:
%sql
drop table retailfashiondata.gold.cust_dim; 
select * from  retailfashiondata.gold.cust_dim;

## Slowly Changing Dimensions

In [0]:
%sql
CREATE TABLE retailfashiondata.gold.cust_dim
AS
SELECT *,
       current_date() AS start_date,
       CAST(NULL AS DATE) AS end_date,
       TRUE AS is_current
FROM retailfashiondata.silvertransformed.cust_dim;

In [0]:


target = DeltaTable.forName(spark,"retailfashiondata.gold.cust_dim")
source = spark.table("retailfashiondata.silvertransformed.cust_dim")

# Deduplicate source by customer_id, keep latest ingestion_time
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

w = Window.partitionBy("customer_id").orderBy(col("ingestion_time").desc())
source_dedup = source.withColumn("rn", row_number().over(w)).filter(col("rn") == 1).drop("rn")

##SCD 2

target.alias("t").merge(
    source.alias("s"),"t.customer_id=s.customer_id"
).whenMatchedUpdate(condition= "t.is_current='True' and(t.city <> s.city OR t.email <> s.email )",
    set={
        "end_date" : lit(current_date()),
         "is_current" : lit("N")
    }
).whenNotMatchedInsert(
    values={
        "customer_id" : "s.customer_id",
        "age" :"s.age",
        "gender" :"s.gender",
        "city" : "s.city",
        "email" : "s.email",
        "ingestion_time" : "s.ingestion_time",
        "phone_number": "s.phone_number",
        "start_date": lit(current_date()),
        "end_date" : lit(None),
        "is_current" : lit("Y")
    }
).whenNotMatchedBySourceUpdate(condition="is_current='True'",
                               set={
                                   "end_date" : lit(current_date()),
                                   "is_current" : lit("N")
}).execute()



In [0]:
%sql
MERGE WITH SCHEMA EVOLUTION
INTO retailfashiondata.gold.cust_dim as t
USING (
    WITH cte AS (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY customer_id
                   ORDER BY ingestion_time DESC
               ) rn
        FROM retailfashiondata.silvertransformed.cust_dim
    )
    SELECT * EXCEPT(rn)
    FROM cte where rn=1
) AS s
ON t.customer_id = s.customer_id

WHEN MATCHED AND t.is_current = 'Y'
              AND (t.city <> s.city OR t.email <> s.email OR t.age <> s.age OR t.gender <> s.gender)
THEN UPDATE SET
    t.end_date   = current_date(),
    t.is_current = 'N'

WHEN NOT MATCHED
THEN INSERT (
    customer_id, age, gender, city, email, ingestion_time, phone_number, start_date, end_date, is_current
)
VALUES (
    s.customer_id, s.age, s.gender, s.city, s.email, s.ingestion_time, s.phone_number,
    current_date(), NULL, 'Y'
)

WHEN NOT MATCHED BY SOURCE AND t.is_current = 'Y'
THEN UPDATE SET
    t.end_date   = current_date(),
    t.is_current = 'N';


In [0]:
%sql
DESCRIBE HISTORY retailfashiondata.gold.cust_dim;

In [0]:
%sql
SELECT *
FROM retailfashiondata.gold.cust_dim VERSION AS OF 4;

In [0]:
%sql
RESTORE TABLE retailfashiondata.gold.cust_dim TO VERSION AS OF 6;

In [0]:
%sql
Select * FROM retailfashiondata.gold.cust_dim  WHERE is_current ="N" ;
--delete from retailfashiondata.gold.cust_dim;

In [0]:


target = DeltaTable.forName(spark,"retailfashiondata.gold.prod_dim")
source = spark.table("retailfashiondata.silvertransformed.prod_dim")

target.alias("t").merge(
    source.alias("s"),"t.product_id=s.product_id"
).whenMatchedUpdate(condition= "t.is_current='True' and(t.category <> s.category OR t.color <> s.color OR t.size <> s.size OR t.season <> s.season OR t.supplier <> s.supplier OR t.cost_price <> s.cost_price OR t.list_price <> s.list_price)",
    set={
        "end_date" : lit(current_date()),
         "is_current" : lit(False)
    }
).whenNotMatchedInsert(
    values={
        "product_id" : "s.product_id" ,
        "category" : "s.category",
        "color" : "s.color",
        "size" : "s.size",
        "season" : "s.season",
        "supplier" : "s.supplier",
        "cost_price" : "s.cost_price",
        "list_price" : "s.list_price",
        "ingestion_time" :"s.ingestion_time",
        "profit_margin" : "s.profit_margin",
        "start_date": lit(current_date()),
        "end_date" : lit(None),
        "is_current" : lit(True)
    }
).whenNotMatchedBySourceUpdate(condition="is_current='True'",
                               set={
                                   "end_date" : lit(current_date()),
                                   "is_current" : lit(False)
}).execute()



In [0]:
%sql
select * from retailfashiondata.gold.prod_dim;

In [0]:

target = DeltaTable.forName(spark,"retailfashiondata.gold.store_dim")
source = spark.table("retailfashiondata.silvertransformed.store_dim")

target.alias("t").merge(
    source.alias("s"),"t.store_id=s.store_id"
).whenMatchedUpdate(condition= "t.is_current='True' and(t.region <> s.region OR t.store_name <> s.store_name OR t.store_size_m2 <> s.store_size_m2)",
    set={
        "end_date" : lit(current_date()),
         "is_current" : lit(False)
    }
).whenNotMatchedInsert(
    values={
        "store_id" : "s.store_id",
        "store_name" : "s.store_name",
        "region" : "s.region",
        "store_size_m2": "s.store_size_m2",
        "ingestion_time" : "s.ingestion_time",
        "start_date": lit(current_date()),
        "end_date" : lit(None),
        "is_current" : lit(True)
    }
).whenNotMatchedBySourceUpdate(condition="is_current='True'",
                               set={
                                   "end_date" : lit(current_date()),
                                   "is_current" : lit(False)
}).execute()



In [0]:
%sql
UPDATE retailfashiondata.gold.cust_dim
SET start_date = '2020-01-01',
    end_date = '9999-12-31'





In [0]:
%sql
DELETE FROM retailfashiondata.gold.cust_dim
WHERE customer_id
BETWEEN 'C025001' AND 'C025010';


In [0]:
df_store_dim = spark.sql("select * from retailfashiondata.gold.store_dim")

df_cust_dim = spark.sql("select * from retailfashiondata.gold.cust_dim")

df_prod_dim = spark.sql("select * from retailfashiondata.gold.prod_dim")

df_sales_fact = spark.sql("select * from retailfashiondata.gold.sales_fact")


In [0]:
df_sales_fact.show()

In [0]:
df_prod_dim.show()

### JoinS

In [0]:
## join the sales_fact with cust_dim
## here i have use joined using SCD@ logic . 

df_cust_dim= (df_cust_dim.withColumnRenamed("ingestion_time","cust_ingestion_time")
                        .withColumnRenamed("customer_id","cust_key"))

sales_cust_joined = df_sales_fact.join(
    df_cust_dim,
    (df_sales_fact.customer_id == df_cust_dim.cust_key) &
    (df_sales_fact.date >= df_cust_dim.start_date) &
    (df_sales_fact.date <= df_cust_dim.end_date),"left"
)

display(sales_cust_joined)

In [0]:
## join the sales_fact with prod_dim
## here i have use joined using SCD@ logic . 

df_prod_dim= (df_prod_dim.withColumnRenamed("ingestion_time","prod_ingestion_time")
                        .withColumnRenamed("product_id","prod_id"))

sales_prod_joined = df_sales_fact.join(
    df_prod_dim,
    (df_sales_fact.product_id == df_prod_dim.prod_id) & 
    (df_sales_fact.date >= df_prod_dim.start_date) &
    (df_sales_fact.date <= df_prod_dim.end_date),"left"

)

display(sales_prod_joined)

In [0]:
## join the sales_fact with store_dim
## here i have use joined using SCD@ logic . 

sales_store_joined = df_sales_fact.join(
    df_store_dim,
    (df_sales_fact.store_id == df_store_dim.store_id) & 
    (df_sales_fact.date >= df_store_dim.start_date) &
    (df_sales_fact.date <= df_store_dim.end_date),"left"

)

display(sales_store_joined)

### KPI VIEWS

In [0]:
## calulate sales_amount

sales_enriched= (sales_prod_joined.withColumn("Sales_amount", round(col("quantity")*col("list_price")*(1-col("discount")), 2))
                            .withColumn("Profit", round(col("quantity") * (col("list_price") - col("cost_price")) * (1 - col("discount")), 2))
                            .withColumn("net_sales_amount", when(col("returned") == True, 0).otherwise(col("Sales_amount")))
                            .withColumn("net_profit",when(col("returned") == True, 0).otherwise(col("Profit")))
                            )
display(sales_enriched)

(sales_enriched.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("retailfashiondata.gold.sales_fact_enriched"))



In [0]:
sales_by_category= (sales_enriched.groupBy("category")
                  .agg(round(sum(col("net_sales_amount")), 2).alias("Total_Sales"))
                  .filter(col("Total_Sales")!=0)
)
display(sales_by_category)

In [0]:
enriched_cust_joined= sales_enriched.join(sales_cust_joined,"customer_id")

sales_by_city = enriched_cust_joined.groupBy("City").agg(round(sum(col("Sales_amount")), 2).alias("Total_sales"))
                                    
display(sales_by_city)

In [0]:
sales_by_gender = enriched_cust_joined.groupBy("Gender").agg(round(sum(col("Sales_amount")), 2).alias("Total_sales"))
                                    
display(sales_by_gender)